# «Вдвоём»: дообучение модели помощника

Нужно: **Среда выполнения → Сменить среду → GPU T4**, затем **Среда выполнения → Выполнить все**. Большое обучение — около 1,5–2 ч на T4.

**Если сессия оборвётся** (уснул компьютер, закрылась вкладка) — просто снова «Выполнить все»: точки сохранения лежат на Google Drive (`vdvoem-train`), обучение продолжится с последней. Готовая модель сама уходит в релиз на GitHub — скачивать ничего не нужно. Для надёжности лучше не давать компьютеру засыпать (на бесплатном Colab закрытая вкладка ≈ отключение через ~1,5 ч).

**До запуска:** в **Секретах** Colab (значок ключа слева) добавь `GH_TOKEN` — токен GitHub с правом *Contents: Read and write* на репозиторий `vdvoem` — и включи ему «Доступ к блокноту». В конце модель (≈1,9 ГБ) сама загрузится в релиз `lora`.

Дальше сервер сам скачает адаптер, проверит модель на запросах, которых она не видела, и включит её, только если она лучше текущей.

**Живые запросы (по желанию):** если Claude прислал файл `live.jsonl` — перетащи его в панель «Файлы» слева перед «Выполнить все». Он не публикуется: тексты остаются у тебя и в Colab.

In [ ]:
BASE = "unsloth/Qwen2.5-3B-Instruct"   # совпадает с qwen2.5:3b на сервере
HF_BASE = "Qwen/Qwen2.5-3B-Instruct"
REPO = "vsershov21-spec/vdvoem"
BIG = True   # большое обучение: 14 000 примеров, LoRA r=32, 3 эпохи (~1.5–2 ч на T4, ~25 мин на A100). False — быстрое (6 000, r=16, 2 эпохи)
N = 14000 if BIG else 6300
# точки сохранения — на Google Drive: если сессия оборвётся (уснул ноутбук, закрылась вкладка), «Выполнить все»
# ещё раз продолжит с последней точки, а не с нуля. Разреши доступ к Drive в окошке.
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/vdvoem-train/" + ("big" if BIG else "fast")
!mkdir -p "$OUT"
!pip install -q unsloth
!git clone -q --depth 1 https://github.com/$REPO repo
!N=$N python repo/deploy/train/gen.py data

In [ ]:
import json, torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

model, tok = FastLanguageModel.from_pretrained(BASE, max_seq_length=2048, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=32 if BIG else 16, lora_alpha=64 if BIG else 32, lora_dropout=0, use_gradient_checkpointing="unsloth",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
load = lambda p: [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
train, valid, test = load("data/train.jsonl"), load("data/valid.jsonl"), load("data/test.jsonl")
# живые запросы пары (файл live.jsonl из бота: перетащи его в «Файлы» слева до запуска) — ×3, это самые ценные примеры
import os
if os.path.exists("live.jsonl"):
    live = load("live.jsonl")
    train += live * 3
    print("живых примеров:", len(live))
else:
    print("live.jsonl нет — учимся только на синтетике")
fmt = lambda r: {"text": tok.apply_chat_template(r["messages"], tokenize=False)}
trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=Dataset.from_list(train).map(fmt), eval_dataset=Dataset.from_list(valid).map(fmt),
    args=SFTConfig(dataset_text_field="text", per_device_train_batch_size=8, gradient_accumulation_steps=2, num_train_epochs=3 if BIG else 2,
                   learning_rate=2e-4, warmup_ratio=0.05, lr_scheduler_type="cosine", logging_steps=20, eval_strategy="epoch",
                   output_dir=OUT, save_strategy="steps", save_steps=150, save_total_limit=2, report_to="none"))
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
import glob as _g
resume = bool(_g.glob(OUT + "/checkpoint-*"))
print("продолжаю с точки сохранения" if resume else "начинаю с нуля")
trainer.train(resume_from_checkpoint=resume or None)

In [ ]:
# проверка на test — фразы и названия, которых не было в обучении
FastLanguageModel.for_inference(model)
def ask(msgs):
    ids = tok.apply_chat_template(msgs[:-1], add_generation_prompt=True, return_tensors="pt").to("cuda")  # a dialogue keeps its first turn
    out = model.generate(ids, max_new_tokens=120, do_sample=False)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()
ok_i = ok_all = 0
bad = []
for r in test:
    want = json.loads(r["messages"][-1]["content"])
    try: got = json.loads(ask(r["messages"]))
    except Exception: got = {}
    ok_i += got.get("i") == want["i"]
    ok_all += got == want
    if got != want and len(bad) < 15: bad.append((r["messages"][-2]["content"], want, got))
print(f"тип запроса верно: {ok_i}/{len(test)} · полностью верно: {ok_all}/{len(test)}")
for b in bad: print(" ✗", b)

In [ ]:
# модель + адаптер → один файл GGUF q4_k_m (Ollama 0.40+ не принимает LoRA отдельно) → релиз «lora» на GitHub
import glob, os, requests
model.save_pretrained_gguf("merged", tok, quantization_method="q4_k_m")
f = [p for p in glob.glob("merged*/**/*.gguf", recursive=True) + glob.glob("*.gguf") if "q4_k_m" in p.lower()][0]
print(f, round(os.path.getsize(f) / 1e9, 2), "ГБ")
from google.colab import userdata
token = userdata.get("GH_TOKEN")  # ошибка здесь = нет секрета GH_TOKEN или выключен «Доступ к блокноту»
H = {"Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json"}
api = f"https://api.github.com/repos/{REPO}/releases"
r = requests.get(f"{api}/tags/lora", headers=H)
rel = r.json() if r.ok else requests.post(api, headers=H, json={"tag_name": "lora", "name": "Дообученная модель помощника",
    "body": "qwen2.5:3b, дообученная понимать запросы. Ставится на сервер deploy/train/install-lora.sh", "prerelease": True}).json()
for a in rel.get("assets", []):
    if a["name"] == "vdvoem-intent-q4_k_m.gguf":
        requests.delete(a["url"], headers=H)
up = rel["upload_url"].split("{")[0] + "?name=vdvoem-intent-q4_k_m.gguf"
with open(f, "rb") as fh:
    res = requests.post(up, headers={**H, "Content-Type": "application/octet-stream"}, data=fh)
print("загружено" if res.ok else res.text[:500])